# Adaptive Agent Search REPL: Hands-On LLM Repository
This notebook serves as an **interactive, persistent REPL** designed for an agent (or researcher) to experiment with different repository search strategies, evaluate search effectiveness, and adapt its search tactics based on previous failures.

### Why an Adaptive Search Trace?
Traditional search treats queries in isolation. An adaptive agent:
1. **Remembers past search attempts** within the session.
2. **Diagnoses failure modes** (e.g., query too specific, noisy matches, wrong search modality, searching raw JSON instead of code).
3. **Formulates an explicit lesson** from each attempt.
4. **Pivots search strategy** (e.g., Lexical Code $\to$ Fuzzy Headings $\to$ Structural Inspection) to locate precise evidence.

---


## 1. Setup & Environment
We import lightweight standard-library tools (`difflib`, `re`, `json`, `pathlib`) and define the target repository path (`Hands-On-Large-Language-Models`).


In [1]:
import os
import re
import json
import difflib
import time
from datetime import datetime
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import List, Dict, Optional, Any

# Target repository path
REPO_PATH = Path("Hands-On-Large-Language-Models")
assert REPO_PATH.exists(), f"Target repository '{REPO_PATH}' not found in current workspace."
print(f"Target repository verified: {REPO_PATH.resolve()}")


Target repository verified: /Users/jaswanthibanoth/Desktop/jev/Hands-On-Large-Language-Models


## 2. Repository Indexer: Clean Structure & Cell Extraction
Jupyter notebooks (`.ipynb`) are stored as JSON files. Searching raw `.ipynb` files with naive regex often floods results with JSON metadata, base64 images, or execution counts.

Here, we build a clean in-memory index that parses:
- **Files**: All documentation, code, and configuration files.
- **Headings**: Extracted markdown titles and section headings (`#`, `##`, `###`) mapped to file paths and cell/line numbers.
- **Notebook Cells**: Parsed into discrete `markdown` and `code` cells so searches return clean, readable code and text snippets.


In [2]:
class RepoIndex:
    """Structured index separating files, notebook cells, and headings."""
    def __init__(self, repo_dir: Path):
        self.repo_dir = repo_dir
        self.files: List[str] = []
        self.headings: List[Dict[str, Any]] = [] # {heading, file, cell_or_line, level}
        self.cells: List[Dict[str, Any]] = []    # {file, cell_idx, cell_type, content}
        self.build_index()

    def build_index(self):
        for p in sorted(self.repo_dir.rglob("*")):
            if ".git" in p.parts:
                continue
            if not p.is_file():
                continue

            rel_path = str(p.relative_to(self.repo_dir))
            self.files.append(rel_path)

            # Handle Jupyter Notebooks
            if p.suffix == ".ipynb":
                try:
                    with open(p, "r", encoding="utf-8") as f:
                        data = json.load(f)
                    for c_idx, cell in enumerate(data.get("cells", [])):
                        c_type = cell.get("cell_type", "unknown")
                        source = "".join(cell.get("source", []))
                        self.cells.append({
                            "file": rel_path,
                            "cell_idx": c_idx,
                            "cell_type": c_type,
                            "content": source
                        })
                        if c_type == "markdown":
                            for line in source.splitlines():
                                stripped = line.strip()
                                if stripped.startswith("#"):
                                    match = re.match(r"^(#+)\s*(.*)", stripped)
                                    if match:
                                        level = len(match.group(1))
                                        title = match.group(2).strip()
                                        self.headings.append({
                                            "heading": title,
                                            "file": rel_path,
                                            "cell_idx": c_idx,
                                            "level": level
                                        })
                except Exception as e:
                    # In case of corrupted notebook
                    pass

            # Handle Markdown, Python, and text files
            elif p.suffix in [".md", ".py", ".txt", ".yml", ".yaml"]:
                try:
                    raw = p.read_bytes()
                    text = None
                    for enc in ["utf-8", "utf-16", "utf-16-le", "latin-1"]:
                        try:
                            text = raw.decode(enc)
                            break
                        except Exception:
                            continue
                    if text:
                        self.cells.append({
                            "file": rel_path,
                            "cell_idx": 0,
                            "cell_type": "file",
                            "content": text
                        })
                        for l_idx, line in enumerate(text.splitlines()):
                            stripped = line.strip()
                            if stripped.startswith("#"):
                                match = re.match(r"^(#+)\s*(.*)", stripped)
                                if match:
                                    self.headings.append({
                                        "heading": match.group(2).strip(),
                                        "file": rel_path,
                                        "cell_idx": l_idx,
                                        "level": len(match.group(1))
                                    })
                except Exception:
                    pass

repo_idx = RepoIndex(REPO_PATH)
print(f"Index built successfully:")
print(f"  • Files indexed: {len(repo_idx.files)}")
print(f"  • Markdown headings cataloged: {len(repo_idx.headings)}")
print(f"  • Text/Code cells indexed: {len(repo_idx.cells)}")


Index built successfully:
  • Files indexed: 83
  • Markdown headings cataloged: 179
  • Text/Code cells indexed: 531


## 3. The Search & Inspection Toolbox
The agent has three primary information retrieval tools, plus a context viewer:

1. `inspect_structure(path_prefix)`: Explores the directory tree and chapter files.
2. `fuzzy_search(query, target)`: Matches approximate keywords against headings or file paths using `difflib` similarity ratios.
3. `lexical_search(query, scope, file_filter)`: Regex/keyword search across parsed text or code cells, returning exact matches with context snippets.
4. `view_cell(file, cell_idx)`: Opens the full content of a target cell or file section for deep inspection.


In [3]:
RG_PATH = shutil.which("rg") or "/Users/jaswanthibanoth/Library/Application Support/arkenOS/agent/bin/rg"

def ripgrep_search(query: str, path_filter: Optional[str] = None, max_results: int = 8, context_lines: int = 1) -> List[Dict[str, Any]]:
    """Fast raw text search using ripgrep (rg) CLI."""
    if not Path(RG_PATH).exists():
        return [{"error": f"ripgrep binary not found at {RG_PATH}"}]

    cmd = [RG_PATH, "-n", "-C", str(context_lines), "--max-count", str(max_results)]
    if path_filter:
        cmd.extend(["-g", f"*{path_filter}*"])
    cmd.extend([query, str(REPO_PATH)])

    res = subprocess.run(cmd, capture_output=True, text=True)
    results = []
    current_hit = None

    for line in res.stdout.splitlines():
        if line.startswith(str(REPO_PATH)):
            parts = line.split(":", 2)
            if len(parts) >= 3:
                rel = str(Path(parts[0]).relative_to(REPO_PATH))
                results.append({
                    "file": rel,
                    "line": int(parts[1]) if parts[1].isdigit() else 0,
                    "matched_snippet": parts[2].strip()
                })
        elif results and line.strip() and not line.startswith("--"):
            # Append context line
            results[-1]["matched_snippet"] += f"\n{line.strip()}"

    return results[:max_results]


def inspect_structure(path_prefix: Optional[str] = None, max_items: int = 25) -> List[str]:
    """Inspect file paths matching an optional prefix (e.g. 'chapter08' or 'bonus')."""
    matches = [f for f in repo_idx.files if not path_prefix or f.startswith(path_prefix)]
    return matches[:max_items]


def fuzzy_search(query: str, target: str = "headings", top_k: int = 5, min_score: float = 0.35) -> List[Dict[str, Any]]:
    """Fuzzy match against headings or file names."""
    results = []
    q_norm = query.lower().strip()

    if target == "headings":
        for h in repo_idx.headings:
            h_text = h["heading"]
            h_norm = h_text.lower()
            score = difflib.SequenceMatcher(None, q_norm, h_norm).ratio()
            if q_norm in h_norm:
                score = max(score, 0.85)
            if score >= min_score:
                results.append({
                    "score": round(score, 3),
                    "match": h_text,
                    "file": h["file"],
                    "cell_idx": h["cell_idx"],
                    "level": h["level"]
                })
    elif target == "files":
        for f in repo_idx.files:
            f_norm = f.lower()
            score = difflib.SequenceMatcher(None, q_norm, f_norm).ratio()
            if q_norm in f_norm:
                score = max(score, 0.85)
            if score >= min_score:
                results.append({
                    "score": round(score, 3),
                    "match": f,
                    "file": f,
                    "cell_idx": 0,
                    "level": 0
                })

    results.sort(key=lambda x: x["score"], reverse=True)
    return results[:top_k]


def lexical_search(
    query: str, 
    scope: str = "all",          # 'all', 'code', 'markdown'
    file_filter: Optional[str] = None, 
    max_results: int = 6,
    ignore_case: bool = True
) -> List[Dict[str, Any]]:
    """Search for keywords or regex patterns in notebook cells and documents."""
    results = []
    flags = re.IGNORECASE if ignore_case else 0
    try:
        pattern = re.compile(query, flags)
    except re.error:
        pattern = re.compile(re.escape(query), flags)

    for cell in repo_idx.cells:
        if file_filter and not re.search(file_filter, cell["file"], re.IGNORECASE):
            continue
        if scope == "code" and cell["cell_type"] != "code":
            continue
        if scope == "markdown" and cell["cell_type"] not in ["markdown", "file"]:
            continue

        content = cell["content"]
        match = pattern.search(content)
        if match:
            lines = content.splitlines()
            hit_lines = []
            for idx, line in enumerate(lines):
                if pattern.search(line):
                    start = max(0, idx - 1)
                    end = min(len(lines), idx + 2)
                    context_snippet = "\n".join(lines[start:end])
                    hit_lines.append(context_snippet)
                    if len(hit_lines) >= 2:
                        break

            results.append({
                "file": cell["file"],
                "cell_idx": cell["cell_idx"],
                "cell_type": cell["cell_type"],
                "matched_snippet": "\n---\n".join(hit_lines)[:400]
            })
            if len(results) >= max_results:
                break

    return results


def view_cell(file: str, cell_idx: int) -> str:
    """Retrieve the complete content of a specific cell or file."""
    for cell in repo_idx.cells:
        if cell["file"] == file and cell["cell_idx"] == cell_idx:
            header = f"=== [{cell['file']} | Cell {cell_idx} ({cell['cell_type']})] ==="
            return f"{header}\n{cell['content']}"
    return f"Cell {cell_idx} in {file} not found."


## 4. Persistent Search Trace & Memory Store
An adaptive agent must not repeat failed queries blindly. It needs:
1. **A persistent trace of actions**: query, method, files returned, and timestamps.
2. **Ground-truth feedback / evaluation**: Did the result help answer the question?
3. **An explicit lesson learned**: What went wrong and what tactical pivot to make next.

Below is `SearchTraceLog`, which keeps this history in memory and can export to / load from JSON.


In [4]:
@dataclass
class SearchRecord:
    step: int
    timestamp: str
    question: str
    method: str
    query: str
    hits_count: int
    top_files: List[str]
    helped_answer: bool
    assessment: str
    lesson_learned: str

class SearchTraceLog:
    """Maintains persistent memory of past search attempts and adaptive lessons."""
    def __init__(self, storage_path: str = "search_trace_log.json"):
        self.storage_path = Path(storage_path)
        self.records: List[SearchRecord] = []
        self.load()

    def record_attempt(
        self,
        question: str,
        method: str,
        query: str,
        results: List[Dict[str, Any]],
        helped_answer: bool,
        assessment: str,
        lesson_learned: str
    ) -> SearchRecord:
        top_files = list(dict.fromkeys([r.get("file", "") for r in results if "file" in r]))[:3]
        rec = SearchRecord(
            step=len(self.records) + 1,
            timestamp=datetime.now().strftime("%H:%M:%S"),
            question=question,
            method=method,
            query=query,
            hits_count=len(results),
            top_files=top_files,
            helped_answer=helped_answer,
            assessment=assessment,
            lesson_learned=lesson_learned
        )
        self.records.append(rec)
        self.save()
        return rec

    def display_history(self, filter_question: Optional[str] = None):
        """Print a formatted trace of all searches and lessons learned."""
        records = [r for r in self.records if not filter_question or filter_question.lower() in r.question.lower()]
        if not records:
            print("No search history recorded yet.")
            return

        print(f"\n{'='*90}")
        print(f"{'STEP':<5} | {'METHOD':<15} | {'STATUS':<7} | {'QUERY':<25} | {'FILES / HITS':<30}")
        print(f"{'='*90}")
        for r in records:
            status = "✅ HELPED" if r.helped_answer else "❌ FAILED"
            files_str = f"{r.hits_count} hits in {', '.join(Path(f).name for f in r.top_files)}" if r.top_files else "0 hits"
            print(f"{r.step:<5} | {r.method:<15} | {status:<7} | {r.query[:25]:<25} | {files_str[:30]:<30}")
            print(f"  ↳ Assessment: {r.assessment}")
            print(f"  ↳ Lesson:     {r.lesson_learned}")
            print(f"{'-'*90}")

    def get_recent_lessons(self, n: int = 3) -> List[str]:
        """Extract recent lessons to inform the agent's next action."""
        return [f"Step {r.step} ({r.method} on '{r.query}'): {r.lesson_learned}" for r in self.records[-n:]]

    def save(self):
        try:
            with open(self.storage_path, "w", encoding="utf-8") as f:
                json.dump([asdict(r) for r in self.records], f, indent=2)
        except Exception:
            pass

    def load(self):
        if self.storage_path.exists():
            try:
                with open(self.storage_path, "r", encoding="utf-8") as f:
                    data = json.load(f)
                    self.records = [SearchRecord(**d) for d in data]
            except Exception:
                self.records = []

# Instantiate persistent trace
search_trace = SearchTraceLog()
print(f"Search trace initialized. Loaded {len(search_trace.records)} existing records from disk.")


Search trace initialized. Loaded 0 existing records from disk.


## 5. The Adaptive Search Loop
Now we connect the components into an observable loop:
1. **Receive Question**: Define the target information need.
2. **Review Memory**: Check prior lessons from the trace before querying.
3. **Execute Search**: Run the selected search tool.
4. **Evaluate Evidence**: Assess whether the returned snippets answer the question completely.
5. **Formulate Lesson & Adapt**: If insufficient, diagnose the root cause (e.g. wrong vocabulary, wrong scope, missing context) and record what to adjust next.


In [ ]:
class MemoryBinIndexManager:
    """Manages loading, matching, and telemetry updating for persistent memory-bin indexes."""
    def __init__(self, base_dir: Path = Path("memory-bin/indexes")):
        self.base_dir = base_dir
        self.alias_index_path = self.base_dir / "regex_alias_index.json"
        self.alias_data = self.load_regex_alias_index()

    def load_regex_alias_index(self) -> Dict[str, Any]:
        if self.alias_index_path.exists():
            with open(self.alias_index_path, "r", encoding="utf-8") as f:
                return json.load(f)
        return {"aliases": []}

    def match_alias_index(self, question: str) -> Optional[Dict[str, Any]]:
        q_tokens = set(re.findall(r'[a-zA-Z0-9_\-]+', question.lower()))
        best_match = None
        best_score = 0.0
        best_reason = ""

        for entry in self.alias_data.get("aliases", []):
            aliases = entry.get("aliases", [])
            key_tokens = entry.get("key_tokens", [])
            score = 0.0
            reasons = []

            for alias in aliases:
                a_toks = set(re.findall(r'[a-zA-Z0-9_\-]+', alias.lower()))
                common = q_tokens.intersection(a_toks)
                if common:
                    overlap_ratio = len(common) / len(a_toks)
                    if overlap_ratio > score:
                        score = overlap_ratio
                        reasons.append(f"Overlap with alias '{alias}' on {common}")
                sim = difflib.SequenceMatcher(None, question.lower(), alias.lower()).ratio()
                if sim > score:
                    score = sim
                    reasons.append(f"Fuzzy match with alias '{alias}' (sim={sim:.2f})")

            for kt in key_tokens:
                if kt.lower() in question.lower():
                    score = max(score, 0.85)
                    reasons.append(f"Key token '{kt}' found in question")

            if score > best_score and score >= 0.40:
                best_score = score
                best_match = entry
                best_reason = "; ".join(reasons[-2:])

        if best_match:
            times_used = best_match.get("times_used", 0)
            return {
                "index_consulted": "regex_alias_index.json",
                "matched_concept": best_match["concept"],
                "aliases_retrieved": best_match["aliases"],
                "regex_retrieved": best_match["regex"],
                "target_chapter": best_match["target_chapter"],
                "anchor_cell": best_match.get("anchor_cell"),
                "primary_notebook": best_match.get("primary_notebook"),
                "original_query": question,
                "adapted_query": best_match["regex"],
                "confidence": round(best_score, 3),
                "reason": best_reason or f"Matched concept '{best_match['concept']}'",
                "times_previously_used": times_used,
                "index_reused": times_used > 0
            }
        return None

    def record_usage(self, concept: str, helped: bool):
        for entry in self.alias_data.get("aliases", []):
            if entry["concept"] == concept:
                entry["times_used"] = entry.get("times_used", 0) + 1
                if helped:
                    entry["successful_uses"] = entry.get("successful_uses", 0) + 1
                    entry["tested"] = True
                entry["updated_at"] = datetime.now().isoformat()
                break
        with open(self.alias_index_path, "w", encoding="utf-8") as f:
            json.dump(self.alias_data, f, indent=2)


In [5]:
class AdaptiveSearchAgent:
    """Simulated adaptive search agent that uses past trace feedback to adapt queries."""
    def __init__(self, trace_log: SearchTraceLog):
        self.trace = trace_log

    def search_step(
        self,
        question: str,
        method: str,
        query: str,
        scope: str = "all",
        file_filter: Optional[str] = None
    ) -> List[Dict[str, Any]]:
        print(f"\n--- [Executing Step {len(self.trace.records) + 1}] ---")
        print(f"Question: \"{question}\"")
        print(f"Method:    {method} | Query: \"{query}\" | Scope: {scope}")

        # 1. Execute chosen search tool
        if method == "inspect":
            hits = [{"file": f, "matched_snippet": f} for f in inspect_structure(query)]
        elif method == "fuzzy_headings":
            hits = fuzzy_search(query, target="headings")
        elif method == "fuzzy_files":
            hits = fuzzy_search(query, target="files")
        elif method == "ripgrep":
            hits = ripgrep_search(query, path_filter=file_filter)
        elif method == "lexical":
            hits = lexical_search(query, scope=scope, file_filter=file_filter)
        else:
            raise ValueError(f"Unknown search method: {method}")

        print(f"Results:   Returned {len(hits)} hit(s).")
        for i, h in enumerate(hits[:3]):
            f_name = Path(h.get('file', '')).name
            snippet = h.get('matched_snippet') or h.get('match') or ''
            snippet_clean = ' '.join(snippet.split())[:110]
            print(f"  [{i+1}] {f_name} -> {snippet_clean}...")
        return hits

    def evaluate_and_record(
        self,
        question: str,
        method: str,
        query: str,
        hits: List[Dict[str, Any]],
        helped_answer: bool,
        assessment: str,
        lesson_for_next_time: str
    ):
        rec = self.trace.record_attempt(
            question=question,
            method=method,
            query=query,
            results=hits,
            helped_answer=helped_answer,
            assessment=assessment,
            lesson_learned=lesson_for_next_time
        )
        print(f"Feedback recorded: {'[HELPED]' if helped_answer else '[FAILED]'}")
        print(f"Lesson saved: \"{lesson_for_next_time}\"")
        return rec

agent = AdaptiveSearchAgent(search_trace)
print("AdaptiveSearchAgent ready.")


AdaptiveSearchAgent ready.


## 6. Experiment 1: Locating Conversation Summary Memory
### Target Information Need:
> *"Where and how does the repository implement conversation summary memory, and what exact LangChain class is used?"*

Watch how the agent:
1. First attempts a naive lexical query in markdown, which fails or returns noisy text.
2. Formulates a lesson: *The exact class name is likely in code cells or headings, not prose.*
3. Pivots to `fuzzy_search` on headings to identify the chapter.
4. Narrows scope to `lexical` code search within that specific chapter.
5. Inspects the exact cell with `view_cell` to verify the code parameters.


In [6]:
QUESTION_1 = "Where and how does the repository implement conversation summary memory, and what LangChain class is used?"

# --- ATTEMPT 1: Naive broad lexical search in markdown ---
print(">>> Agent checks prior lessons: none for this question yet.")
hits_1 = agent.search_step(
    question=QUESTION_1,
    method="lexical",
    query="Conversation Summary Memory",
    scope="markdown"
)

# Evaluation: No hits! The heading in the notebook is 'ConversationSummary' without the spaces/words 'Memory'.
agent.evaluate_and_record(
    question=QUESTION_1,
    method="lexical",
    query="Conversation Summary Memory",
    hits=hits_1,
    helped_answer=False,
    assessment="0 hits found. The exact phrase 'Conversation Summary Memory' was not used verbatim in markdown text.",
    lesson_for_next_time="Query was too specific and verbose. Switch to fuzzy heading search with core keywords 'ConversationSummary' to locate the chapter first."
)


>>> Agent checks prior lessons: none for this question yet.

--- [Executing Step 1] ---
Question: "Where and how does the repository implement conversation summary memory, and what LangChain class is used?"
Method:    lexical | Query: "Conversation Summary Memory" | Scope: markdown
Results:   Returned 0 hit(s).
Feedback recorded: [FAILED]
Lesson saved: "Query was too specific and verbose. Switch to fuzzy heading search with core keywords 'ConversationSummary' to locate the chapter first."


In [7]:
# --- ATTEMPT 2: Adapting based on Lesson 1 (Fuzzy Headings) ---
print(">>> Recent Lessons Reviewed:")
for l in search_trace.get_recent_lessons(1):
    print(" ", l)

hits_2 = agent.search_step(
    question=QUESTION_1,
    method="fuzzy_headings",
    query="ConversationSummary"
)

# Evaluation: High confidence hit in Chapter 7!
agent.evaluate_and_record(
    question=QUESTION_1,
    method="fuzzy_headings",
    query="ConversationSummary",
    hits=hits_2,
    helped_answer=True,
    assessment="Found heading 'ConversationSummary' in chapter07/Chapter 7 - Advanced Text Generation Techniques and Tools.ipynb (Cell 23).",
    lesson_for_next_time="Now that the target notebook is located, perform a targeted code search or view the cell directly to extract the exact class and parameters."
)


>>> Recent Lessons Reviewed:
  Step 1 (lexical on 'Conversation Summary Memory'): Query was too specific and verbose. Switch to fuzzy heading search with core keywords 'ConversationSummary' to locate the chapter first.

--- [Executing Step 2] ---
Question: "Where and how does the repository implement conversation summary memory, and what LangChain class is used?"
Method:    fuzzy_headings | Query: "ConversationSummary" | Scope: all
Results:   Returned 5 hit(s).
  [1] Chapter 7 - Advanced Text Generation Techniques and Tools.ipynb -> ConversationSummary...
  [2] Chapter 7 - Advanced Text Generation Techniques and Tools.ipynb -> ConversationBuffer...
  [3] Chapter 7 - Advanced Text Generation Techniques and Tools.ipynb -> ConversationBufferMemoryWindow...
Feedback recorded: [HELPED]
Lesson saved: "Now that the target notebook is located, perform a targeted code search or view the cell directly to extract the exact class and parameters."


In [8]:
# --- ATTEMPT 3: Targeted Code Search in Chapter 7 ---
hits_3 = agent.search_step(
    question=QUESTION_1,
    method="lexical",
    query="ConversationSummaryMemory",
    scope="code",
    file_filter="chapter07"
)

agent.evaluate_and_record(
    question=QUESTION_1,
    method="lexical",
    query="ConversationSummaryMemory",
    hits=hits_3,
    helped_answer=True,
    assessment="Found exact LangChain import and initialization in Chapter 7, cell 24.",
    lesson_for_next_time="Two-stage search (fuzzy heading to find file -> scoped code lexical search) successfully solved the query in 3 steps."
)

# Now retrieve and display the ground-truth cell content
target_hit = hits_3[0]
print("\n" + "="*80)
print("GROUND TRUTH VERIFICATION VIA view_cell:")
print("="*80)
print(view_cell(target_hit["file"], target_hit["cell_idx"]))



--- [Executing Step 3] ---
Question: "Where and how does the repository implement conversation summary memory, and what LangChain class is used?"
Method:    lexical | Query: "ConversationSummaryMemory" | Scope: code
Results:   Returned 1 hit(s).
  [1] Chapter 7 - Advanced Text Generation Techniques and Tools.ipynb -> from langchain.memory import ConversationSummaryMemory --- # Define the type of memory we will use memory = Co...
Feedback recorded: [HELPED]
Lesson saved: "Two-stage search (fuzzy heading to find file -> scoped code lexical search) successfully solved the query in 3 steps."

GROUND TRUTH VERIFICATION VIA view_cell:
=== [chapter07/Chapter 7 - Advanced Text Generation Techniques and Tools.ipynb | Cell 33 (code)] ===
from langchain.memory import ConversationSummaryMemory

# Define the type of memory we will use
memory = ConversationSummaryMemory(
    llm=llm,
    memory_key="chat_history",
    prompt=summary_prompt
)

# Chain the LLM, prompt, and memory together
llm_chain =

## 7. Experiment 2: Locating Reasoning Architectures (Tree-of-Thought)
### Target Information Need:
> *"Which chapter discusses Tree-of-Thought reasoning with local models, and what inference runtime is used?"*

Let's observe another multi-step adaptation:
1. Agent searches for "Tree-of-Thought" in headings.
2. Identifies `chapter06`.
3. Checks imports in `chapter06` to discover the inference runtime (`llama_cpp`).


In [9]:
QUESTION_2 = "Which chapter discusses Tree-of-Thought reasoning with local models, and what inference runtime is used?"

# Step 1: Fuzzy search headings
hits_tot = agent.search_step(
    question=QUESTION_2,
    method="fuzzy_headings",
    query="Tree-of-Thought"
)

agent.evaluate_and_record(
    question=QUESTION_2,
    method="fuzzy_headings",
    query="Tree-of-Thought",
    hits=hits_tot,
    helped_answer=True,
    assessment="Located heading 'Tree-of-Thought: Exploring Intermediate Steps' in chapter06/Chapter 6 - Prompt Engineering.ipynb.",
    lesson_for_next_time="Target chapter is confirmed (Chapter 6). Search for imports or model loaders in chapter 6 code cells to identify the local inference runtime."
)

# Step 2: Lexical search for model loader / runtime in Chapter 6
hits_runtime = agent.search_step(
    question=QUESTION_2,
    method="lexical",
    query=r"(Llama|llama_cpp|AutoModelForCausalLM|GGUF)",
    scope="code",
    file_filter="chapter06"
)

agent.evaluate_and_record(
    question=QUESTION_2,
    method="lexical",
    query=r"(Llama|llama_cpp|AutoModelForCausalLM|GGUF)",
    hits=hits_runtime,
    helped_answer=True,
    assessment="Confirmed that Chapter 6 uses `llama_cpp.Llama` with local quantized GGUF models.",
    lesson_for_next_time="Combining heading identification with scoped regex for model libraries resolved both chapter and runtime immediately."
)

# Display the code cell
print("\n" + "="*80)
print(view_cell(hits_runtime[0]["file"], hits_runtime[0]["cell_idx"]))



--- [Executing Step 4] ---
Question: "Which chapter discusses Tree-of-Thought reasoning with local models, and what inference runtime is used?"
Method:    fuzzy_headings | Query: "Tree-of-Thought" | Scope: all
Results:   Returned 4 hit(s).
  [1] Chapter 6 - Prompt Engineering.ipynb -> Tree-of-Thought: Exploring Intermediate Steps...
  [2] Chapter 6 - Prompt Engineering.ipynb -> Zero-shot Chain-of-Thought...
  [3] Chapter 6 - Prompt Engineering.ipynb -> Chain-of-Thought: Think Before Answering...
Feedback recorded: [HELPED]
Lesson saved: "Target chapter is confirmed (Chapter 6). Search for imports or model loaders in chapter 6 code cells to identify the local inference runtime."

--- [Executing Step 5] ---
Question: "Which chapter discusses Tree-of-Thought reasoning with local models, and what inference runtime is used?"
Method:    lexical | Query: "(Llama|llama_cpp|AutoModelForCausalLM|GGUF)" | Scope: code
Results:   Returned 3 hit(s).
  [1] Chapter 6 - Prompt Engineering.ipynb -> # !

## 8. Historical Search Trace Audit
The agent's entire search history is preserved below. Before any new search, an agent can call `search_trace.display_history()` to review which approaches succeeded or failed across different questions.


In [10]:
search_trace.display_history()



STEP  | METHOD          | STATUS  | QUERY                     | FILES / HITS                  
1     | lexical         | ❌ FAILED | Conversation Summary Memo | 0 hits                        
  ↳ Assessment: 0 hits found. The exact phrase 'Conversation Summary Memory' was not used verbatim in markdown text.
  ↳ Lesson:     Query was too specific and verbose. Switch to fuzzy heading search with core keywords 'ConversationSummary' to locate the chapter first.
------------------------------------------------------------------------------------------
2     | fuzzy_headings  | ✅ HELPED | ConversationSummary       | 5 hits in Chapter 7 - Advanced
  ↳ Assessment: Found heading 'ConversationSummary' in chapter07/Chapter 7 - Advanced Text Generation Techniques and Tools.ipynb (Cell 23).
  ↳ Lesson:     Now that the target notebook is located, perform a targeted code search or view the cell directly to extract the exact class and parameters.
------------------------------------------------------

## 9. Summary & Foundation for Downstream Experiments

### What We Have Built:
1. **Clean Structured Indexing**: Parses complex `.ipynb` files into discrete markdown/code cells, eliminating noise from JSON metadata.
2. **Multi-Modal Retrieval Toolbox**:
   - `inspect_structure`: High-level hierarchical discovery.
   - `fuzzy_search`: Resilient against exact keyword mismatches across chapter headings and filenames.
   - `lexical_search`: High-precision regex search scoped to `code`, `markdown`, or specific chapters.
   - `view_cell`: Verification tool to read full implementations in context.
3. **Stateful Search Trace (`SearchTraceLog`)**:
   - Stores each step's query, method, hit count, top files, pass/fail status, and diagnostic lesson.
   - Persists to `search_trace_log.json` across kernel restarts.
4. **Adaptive Feedback Loop**:
   - Explicitly demonstrates how failure diagnosis (e.g. *query too verbose* or *need code instead of prose*) dictates the next retrieval move.

### Natural Extensions for Search Experiments:
- **Adding Dense Vector Search (Ch 8)**: Embed cells using `sentence-transformers` and build a FAISS index to compare lexical vs. semantic retrieval.
- **Hybrid Retrieval & Reranking (Ch 8 & 10)**: Implement BM25 + dense candidates reranked by cross-encoders.
- **Automated LLM Policy Loop (Ch 7 & Bonus 9)**: Plug an LLM into the agent loop to automatically generate the next query and diagnose failure notes using the ReAct framework.


In [ ]:
class BenchmarkEvaluator:
    """Independent benchmark evaluator decoupled from the search agent."""
    @staticmethod
    def evaluate(expected_evidence: Dict[str, Any], search_output: Dict[str, Any]) -> Dict[str, Any]:
        hits = search_output.get("hits", [])
        combined_text = " ".join(h.get("matched_snippet", "") for h in hits).lower()
        hit_files = [h.get("file", "") for h in hits]

        required_all = [t.lower() for t in expected_evidence.get("required_all", [])]
        required_any = [t.lower() for t in expected_evidence.get("required_any", [])]
        expected_files = expected_evidence.get("expected_files", [])

        matched_all = [t for t in required_all if t in combined_text]
        missing_all = [t for t in required_all if t not in combined_text]
        matched_any = [t for t in required_any if t in combined_text]
        matched_files = [f for f in expected_files if any(f in hf for hf in hit_files)]

        all_ok = (len(missing_all) == 0)
        any_ok = (len(matched_any) > 0) if required_any else True
        file_ok = (len(matched_files) > 0) if expected_files else True

        passed = all_ok and any_ok and file_ok
        reasons = []
        if not all_ok:
            reasons.append(f"Missing required tokens: {missing_all}")
        if required_any and not any_ok:
            reasons.append(f"None of required_any tokens found ({required_any})")
        if expected_files and not file_ok:
            reasons.append(f"No hits from expected files ({expected_files})")

        return {
            "passed": passed,
            "matched_all": matched_all,
            "missing_all": missing_all,
            "matched_any": matched_any,
            "matched_files": matched_files,
            "reason": "All expected evidence criteria satisfied." if passed else "; ".join(reasons)
        }
